In [ ]:
from pyspark.sql import functions as F

#Environment passaed by Databricks Job
environment = dbutils.widgets.get("environment")

if environment not in ("dev", "staging", "prod"):
    raise ValueError(f"Invalid environment: {environment}")

schema = f"opensource_{environment}.github_analytics"

#Check that the main layers contain recors
tables = [
    "bronze_github_repositories",
    "silver_staging_github_repositories",
    "silver_github_repositories",
    "gold_github_repositories",
    "gold_github_repository_metrics"
]

for table_name in tables:
    count = spark.table(f"{schema}.{table_name}").count()
    print(f"{table_name}: {count} records")

    if count == 0:
        raise ValueError(f"Data Quality FAILED: {table_name} is empty")

#Check SCD Type 2 consistency
gold = spark.table(f"{schema}.gold_github_repositories")

active = gold.filter(F.col("current") == True)

duplicates = (
    active
    .groupBy("id", "organization")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

if duplicates > 0:
    raise ValueError(f"Data Quality FAILED: {duplicates} duplicate active repositories")

invalid_versions = active.filter(
    F.col("valid_to").isNotNull()
).count()

if invalid_versions > 0:
     raise ValueError(f"Data Quality FAILED: {invalid_versions} active SCD2 records have valid_to populated")

print("All Data Quality Checks PASSED!")